In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mod ujian tempatan Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Versi pakej</b></summary>

Kod dalam halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Gunakan mod ujian tempatan (tersedia dengan `qiskit-ibm-runtime` v0.22.0 atau lebih baru) untuk menguji program sebelum menala-halusnya dan menghantar ke perkakasan kuantum sebenar. Selepas menggunakan mod ujian tempatan untuk mengesahkan program anda, satu-satunya perubahan yang perlu anda lakukan ialah menukar nama Backend untuk menjalankannya pada QPU.

Untuk menggunakan mod ujian tempatan, tentukan salah satu backend palsu daripada ``qiskit_ibm_runtime.fake_provider`` atau tentukan Backend Qiskit Aer semasa menginisialisasi primitif Qiskit Runtime atau sesi.

- **Backend palsu**: [Backend palsu](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) dalam ``qiskit_ibm_runtime.fake_provider`` meniru tingkah laku QPU IBM&reg; dengan menggunakan syot kilat QPU. Syot kilat QPU mengandungi maklumat penting tentang QPU, seperti peta gandingan, get asas, dan sifat qubit, yang berguna untuk menguji Transpiler dan melakukan simulasi berhalangan QPU. Model hingar daripada syot kilat digunakan secara automatik semasa simulasi.
- **Simulator Aer**: Simulator daripada [Qiskit Aer](/guides/simulate-with-qiskit-aer) menyediakan simulasi berprestasi lebih tinggi yang boleh mengendalikan Circuit yang lebih besar dan [model hingar kustom](/guides/build-noise-models). Senarai pilihan kaedah simulasi tersedia apabila anda menggunakan `AerSimulator` dalam mod ujian tempatan. Lihat [contoh mod simulasi Clifford](#clifford-sim), yang menunjukkan cara mensimulasikan Circuit Clifford dengan bilangan qubit yang besar secara cekap.

    <details>
    <summary>**Senarai kaedah simulasi yang tersedia daripada Qiskit Aer**</summary>

    Lihat dokumentasi [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) untuk maklumat lanjut.

    * ``"automatic"``: Kaedah simulasi lalai. Pilih kaedah simulasi secara automatik, berdasarkan Circuit dan model hingar.

    * ``"statevector"``: Simulasi vektor keadaan padat yang boleh mengambil sampel hasil pengukuran daripada Circuit *ideal* dengan semua pengukuran di penghujung Circuit. Untuk simulasi berhalangan, setiap tembakan mengambil sampel Circuit berhalangan secara rawak daripada model hingar.

    * ``"density_matrix"``: Simulasi matriks ketumpatan yang boleh mengambil sampel hasil pengukuran daripada Circuit *berhalangan* dengan semua pengukuran di penghujung Circuit.

    * ``"stabilizer"``: Simulator keadaan penstabil Clifford yang cekap yang boleh mensimulasikan Circuit Clifford berhalangan jika semua ralat dalam model hingar juga merupakan ralat Clifford.

    * ``"extended_stabilizer"``: Simulator anggaran untuk Circuit Clifford + T berdasarkan penguraian keadaan kepada keadaan penstabil berperingkat. Bilangan sebutan bertambah dengan bilangan get bukan-Clifford (T).

    * ``"matrix_product_state"``: Simulator vektor keadaan rangkaian tensor yang menggunakan perwakilan Keadaan Hasil Darab Matriks (MPS) untuk keadaan. Ini boleh dilakukan dengan atau tanpa memotong dimensi ikatan MPS, bergantung pada pilihan simulator. Tingkah laku lalai adalah tanpa pemotongan.

    * ``"unitary"``: Simulasi matriks unitari padat bagi Circuit ideal. Ini mensimulasikan matriks unitari Circuit itu sendiri, bukan evolusi keadaan kuantum awal. Kaedah ini hanya boleh mensimulasikan get; ia tidak menyokong pengukuran, penetapan semula, atau hingar.

    * ``"superop"``: Simulasi matriks superoperator padat bagi Circuit ideal atau berhalangan. Ini mensimulasikan matriks superoperator Circuit itu sendiri, bukan evolusi keadaan kuantum awal. Kaedah ini boleh mensimulasikan get dan penetapan semula ideal dan berhalangan, tetapi tidak menyokong pengukuran.

    * ``"tensor_network"``: Simulasi berasaskan rangkaian tensor yang menyokong vektor keadaan dan matriks ketumpatan. Pada masa ini ini hanya tersedia untuk GPU dan dipercepatkan dengan menggunakan API cuQuantum `cuTensorNet`.
    </details>

> **Note:** - Anda boleh menentukan semua pilihan Qiskit Runtime dalam mod ujian tempatan. Walau bagaimanapun, semua pilihan kecuali shots diabaikan apabila dijalankan pada simulator tempatan.
> - Adalah disyorkan untuk memasang Qiskit Aer sebelum menggunakan backend palsu atau simulator Aer dengan menjalankan `pip install qiskit-aer`. Backend palsu menggunakan simulator Aer di sebalik tabir jika tersedia, untuk memanfaatkan prestasinya.


## Contoh backend palsu

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Contoh-contoh AerSimulator
Contoh dengan sesi, tanpa hingar:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Untuk mensimulasikan dengan hingar, tentukan QPU (perkakasan kuantum) dan hantar ke Aer. Aer akan membina model hingar berdasarkan data penentukuran daripada QPU tersebut, dan menginisialisasi Backend Aer dengan model itu. Jika anda lebih suka, anda boleh [membina model hingar](/guides/build-noise-models).

> **Caution:** QPU boleh dipengaruhi oleh pelbagai jenis hingar. Model hingar Qiskit Aer yang digunakan di sini hanya mensimulasikan sebahagian daripadanya dan oleh itu kemungkinan kurang teruk berbanding hingar pada QPU sebenar.
> 
> Untuk butiran tentang ralat yang disertakan apabila menginisialisasi model hingar daripada QPU, lihat rujukan API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Contoh dengan hingar:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Simulasi Clifford
Oleh kerana Circuit Clifford boleh disimulasikan secara cekap dengan hasil yang boleh disahkan, simulasi Clifford adalah alat yang sangat berguna. Untuk contoh yang mendalam, lihat [Simulasi cekap Circuit penstabil dengan primitif Qiskit Aer](/guides/simulate-stabilizer-circuits).

Contoh:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Langkah seterusnya
> **Tip:** - Semak [contoh-contoh primitif](/guides/primitives-examples) yang terperinci.
>     - Baca [Migrasi ke primitif V2](/guides/v2-primitives).
>     - Berlatih dengan primitif melalui [pelajaran Fungsi kos](/learning/courses/variational-algorithm-design/cost-functions) dalam IBM Quantum Learning.
>     - Pelajari cara transpile secara tempatan dalam bahagian [Transpile](/guides/transpile).
>     - Cuba tutorial [Bandingkan tetapan Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).